# Profiling Regional Grid Load and Outages with PROC CHART

## Executive Summary

A grid operations analyst uses PROC CHART to profile a synthetic month of feeder-circuit meter readings — 1,200 readings across four service regions and four generation sources. The notebook walks through vertical bar, horizontal bar, block, and pie charts to summarize the generation mix, compare mean and total delivered load by region, expose the evening demand peak by hour, and rank outage minutes by source — the kind of fast, text-output exploration an energy-and-utilities team runs before committing to a graphical report. Across the full sample, Gas dominates the generation mix at 42.9% of readings, North carries the highest average delivered load (25.2 MWh) and the most total energy, the load curve peaks sharply in the 17:00–21:00 evening window (the hour-18 bin clears 30 MWh, versus roughly 20 MWh off-peak), and Gas accumulates the most outage minutes (1,342 of 3,024 total) — chiefly because it serves the most circuits.

## Data Sources

| Dataset | Rows | Description |
|---------|------|-------------|
| `grid_ops` | 1,200 (synthetic) | One row per feeder-circuit meter reading on a regional grid, generated inline with `call streaminit(20260531)` and `rand()`. The DATA step loop produces 1,200 readings, each carrying a service region, generation source, hour of day, delivered load (MWh), a peak/off-peak flag, and Poisson-distributed outage minutes. |

# Profiling Regional Grid Load and Outages with PROC CHART

PROC CHART produces character-based bar, block, and pie charts directly in the listing — no ODS Graphics device required. That makes it a quick first-pass exploration tool for a grid operations team that wants to *see* the shape of its load and reliability data before building polished GCHART or SGPLOT visuals.

In this notebook we:

1. Generate a synthetic month of feeder-circuit meter readings for a regional grid operator.
2. Chart the **generation mix** (share of readings by source).
3. Compare **mean and total delivered load** across service regions.
4. Expose the **evening demand peak** by hour of day.
5. Use a **block chart** to cross region against generation source.
6. Rank **outage minutes** by source to find the least reliable feed.

Every statement and option below is standard SAS 9.4 PROC CHART syntax.

## Step 1 — Generate the synthetic operations data

The DATA step below fabricates meter readings in a loop of 1,200 iterations. Each reading is assigned a service region, a generation source (Gas dominates, with Wind, Solar, and Hydro making up the rest), and an hour of day. Load is higher in the 17:00–21:00 evening window (and we flag those readings as peak), and we draw outage minutes from a Poisson distribution. `call streaminit` fixes the random seed so the data is reproducible.

The log confirms the result: `grid_ops` is written with **1,200 rows and 7 columns**, and each PROC CHART below reports reading all 1,200 rows.

In [1]:
/* Synthetic feeder-circuit operations for a regional grid operator */
data grid_ops;
    call streaminit(20260531);
    length region $12 source $9 peak_flag $3;
    array regions[4] $12 _temporary_
        ('North','South','East','West');
    do meter_id = 1 to 1200;
        r = ceil(4 * rand('uniform'));
        region = regions[r];
        u = rand('uniform');
        if u < 0.42 then source = 'Gas';
        else if u < 0.70 then source = 'Wind';
        else if u < 0.88 then source = 'Solar';
        else source = 'Hydro';
        hour = floor(24 * rand('uniform'));
        base = 18 + 5 * (region = 'North') + 3 * (region = 'West');
        if hour >= 17 and hour <= 21 then do;
            load_mwh = base + 12 + 6 * rand('normal');
            peak_flag = 'Yes';
        end;
        else do;
            load_mwh = base + 4 * rand('normal');
            peak_flag = 'No';
        end;
        if load_mwh < 0 then load_mwh = 0;
        outage_min = rand('poisson', 2.5);
        output;
    end;
    drop r u base;
run;

NOTE: DATA grid_ops


NOTE: Wrote grid_ops (1200 rows, 7 columns).
NOTE: DATA elapsed:
  wall  0.03 seconds
  cpu   0.03 seconds


## Step 2 — Generation mix

What share of readings does each generation source account for? A vertical bar chart with `TYPE=PERCENT` answers this directly: bar heights are the percentage of all observations falling in each source category. Because `source` is a character variable, PROC CHART treats its values as discrete categories automatically.

In [2]:
proc chart data=grid_ops;
    vbar source / type=percent;
run;


Percent of source

         |              *****       
   40.00 +              *****       
         |              *****       
         |              *****       
         |              *****       
   30.00 +       *****  *****       
         |       *****  *****       
         |       *****  *****       
   20.00 +       *****  *****       
         |       *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
   10.00 +*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
         |                          
         +--------------------------+
          Hydro  Wind    Gas   Solar
                    source




NOTE: PROC CHART data=grid_ops

NOTE: 1200 rows read from dataset 'grid_ops'.


## Step 3 — Mean delivered load by region

Now we switch from counting to summarizing a measurement. Naming `load_mwh` in `SUMVAR=` with `TYPE=MEAN` makes the bar length the average load per region, and we request the statistics columns explicitly: `MEAN` prints the average beside each bar and `CFREQ` adds a cumulative-frequency column.

We also pass `ASCENDING` to order the bars from lowest to highest mean, and the chart honours it: the bars print **South (20.44) → East (21.42) → West (23.67) → North (25.20)**, strictly shortest-to-longest. North carries the highest average delivered load and South the lowest, consistent with the regional offset (+5 MWh North, +3 MWh West) built into the data.

In [3]:
proc chart data=grid_ops;
    hbar region / sumvar=load_mwh type=mean
                  cfreq mean ascending;
run;


Mean of region

region                                                  Mean           N     Percent
                                                                                    
 South  ********************************               20.44       20.44       25.92
  East  **********************************             21.42       41.86       24.42
  West  **************************************         23.67       65.53       24.50
 North  ****************************************       25.20       90.73       25.17




NOTE: PROC CHART data=grid_ops

NOTE: 1200 rows read from dataset 'grid_ops'.


## Step 4 — Total load by region, split by peak window

Here `TYPE=SUM` makes each bar the *total* delivered load for the region rather than the average, so the taller bars mark the regions delivering the most aggregate energy. The bars cluster in the 6,000–8,000 MWh band, with **North** reaching highest — it delivers the most total energy across the month — while the other three regions sit close together just below it.

The statement also requests `SUBGROUP=peak_flag`, splitting each bar by whether the reading fell in the evening peak window. PROC CHART segments each bar with a distinct glyph per subgroup level (`N` for off-peak, `Y` for peak) and prints a symbol legend beneath the chart. The off-peak `N` band dominates every bar — only about 20% of readings fall in the 17:00–21:00 window — but the peak `Y` cap is clearly visible on top of each region, a quick visual of how much of each region's total energy is delivered during peak hours.

In [4]:
proc chart data=grid_ops;
    vbar region / sumvar=load_mwh type=sum
                  subgroup=peak_flag;
run;


Sum of region

         |                     YYYYY
         |YYYYY                YYYYY
         |YYYYY  YYYYY         YYYYY
    6000 +YYYYY  YYYYY  YYYYY  YYYYY
         |YYYYY  YYYYY  YYYYY  NNNNN
         |NNNNN  YYYYY  YYYYY  NNNNN
         |NNNNN  NNNNN  YYYYY  NNNNN
    4000 +NNNNN  NNNNN  NNNNN  NNNNN
         |NNNNN  NNNNN  NNNNN  NNNNN
         |NNNNN  NNNNN  NNNNN  NNNNN
         |NNNNN  NNNNN  NNNNN  NNNNN
    2000 +NNNNN  NNNNN  NNNNN  NNNNN
         |NNNNN  NNNNN  NNNNN  NNNNN
         |NNNNN  NNNNN  NNNNN  NNNNN
         |NNNNN  NNNNN  NNNNN  NNNNN
         |                          
         +--------------------------+
          West   South  East   North
                    region

Symbol  peak_flag
------  ---------
  N     No       
  Y     Yes      




NOTE: PROC CHART data=grid_ops

NOTE: 1200 rows read from dataset 'grid_ops'.


## Step 5 — Load shape across the day

`hour` is continuous, so we pin the binning ourselves with an explicit `MIDPOINTS=` list at 4-hour centers (2, 6, 10, 14, 18, 22). Each bar shows the mean delivered load for readings near that hour. The bar centered on 18 stands out sharply — it is the only bar to rise above the 30 MWh gridline, well clear of the off-peak bins that cluster around 20 MWh — the evening demand peak we built into the data.

In [5]:
proc chart data=grid_ops;
    vbar hour / sumvar=load_mwh type=mean
                midpoints=2 6 10 14 18 22;
run;


Mean of hour

         |                            *****       
   30.00 +                            *****       
         |                            *****       
         |                            *****       
         |                            *****  *****
         |                            *****  *****
   20.00 +*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
   10.00 +*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |                                        
         +----------------------------------------+
            2      6     10     14     18     22  
                            hour




NOTE: PROC CHART data=grid_ops

NOTE: 1200 rows read from dataset 'grid_ops'.


## Step 6 — Region by generation source (block chart)

A BLOCK chart renders a small two-way table as a grid of blocks. With `GROUP=source` and `SUMVAR=load_mwh / TYPE=MEAN`, the chart crosses each region against the generation source serving it, with block height proportional to mean load — a compact way to spot which region/source combinations carry the heaviest average load.

In [6]:
proc chart data=grid_ops;
    block region / sumvar=load_mwh type=mean
                   group=source;
run;


Block chart of Mean of region by source

                          /####\
  /####\                  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  +----+  +----+  +----+  +----+
   West   South    East   North 
             region




NOTE: PROC CHART data=grid_ops

NOTE: 1200 rows read from dataset 'grid_ops'.


## Step 7 — Generation mix as a pie chart

The same source-share information from Step 2, drawn as a pie. PIE with `TYPE=PERCENT` sizes each slice by its percentage of total readings and prints a legend of slice percentages alongside the figure.

In [7]:
proc chart data=grid_ops;
    pie source / type=percent;
run;


Pie chart of source

          source     Percent   Percent  Slice
----------------  ----------  --------  ------------------------------
           Hydro       12.92     12.9%  ####
            Wind       28.17     28.2%  ********
             Gas       42.92     42.9%  +++++++++++++
           Solar       16.00     16.0%  OOOOO




NOTE: PROC CHART data=grid_ops

NOTE: 1200 rows read from dataset 'grid_ops'.


## Step 8 — Outage minutes by source

Finally we rank reliability. `SUMVAR=outage_min` with `TYPE=SUM` totals outage minutes per source. The printed `Sum` column ranks the sources: **Gas accounts for the most total outage minutes (1,342)**, ahead of Wind (846), Solar (476), and Hydro (360), for 3,024 outage-minutes overall. That tracks the generation mix — Gas serves the most readings, so it accumulates the most outage minutes overall.

We pass `DESCENDING` intending to float the worst-performing source to the top. Note that the bars are **not** reordered here — they print in category order (Hydro, Wind, Gas, Solar), the same order as without the option. In this build `DESCENDING` is accepted but does not yet reverse-sort the bars (its sibling `ASCENDING`, used in Step 3, *does* sort), so read the ranking from the printed `Sum` column rather than the bar order.

In [8]:
proc chart data=grid_ops;
    hbar source / sumvar=outage_min type=sum descending;
run;


Sum of source

source                                                   Sum        Cum.     Percent
                                                                     Sum            
 Hydro  ***********                                      360         360       12.92
  Wind  *************************                        846        1206       28.17
   Gas  ****************************************        1342        2548       42.92
 Solar  **************                                   476        3024       16.00




NOTE: PROC CHART data=grid_ops

NOTE: 1200 rows read from dataset 'grid_ops'.


## Interpreting the results

Reading the charts together gives the operations team a fast situational picture over the full 1,200-reading month:

- **Generation mix (Steps 2 and 7).** Gas carries the largest share of readings (42.9%), with Wind second (28.2%), Solar third (16.0%), and Hydro last (12.9%) — the vertical bar and pie tell the same story two ways, a useful sanity check.
- **Load by region (Steps 3 and 4).** The mean-load HBAR, ordered by `ASCENDING`, runs South (20.4 MWh) → East (21.4) → West (23.7) → North (25.2), so North runs the highest average delivered load and South the lowest — consistent with the regional offset built into the data. The SUM chart confirms North also delivers the most total energy — its bar reaches highest of the four — and its `SUBGROUP=peak_flag` split shows the peak-hour `Y` band sitting atop a much larger off-peak `N` base in every region.
- **Daily load shape (Step 5).** The midpoint bar centered on hour 18 is clearly the tallest — the only bar above the 30 MWh gridline, against ~20 MWh in the off-peak bins — confirming the 17:00–21:00 demand peak, exactly where a utility would focus demand-response and capacity planning.
- **Reliability (Step 8).** Totaling outage minutes by source surfaces Gas as the largest contributor of downtime (1,342 of 3,024 minutes), the natural next target for maintenance review — though that mostly reflects Gas serving the most readings.

A note on bar ordering: `ASCENDING` (Step 3) reorders the bars by the chart statistic as expected, while `DESCENDING` (Step 8) is accepted but not yet reversing the bar order in this build — there the ranking is read from the printed statistic column. `SUBGROUP=` segmentation (Step 4) renders with a per-level glyph and a symbol legend.

PROC CHART is character-output only, so for board-ready visuals the team would move these same views to PROC GCHART or PROC SGPLOT. But as a zero-setup first pass over a fresh extract, these charts answer the operational questions — mix, magnitude, and timing — in seconds.